## What has actually been run

**Executed on Colab 2026-08-10** (Ubuntu 22.04.5, 2 vCPU, 12 GB RAM, 88 GB free), through the kernel suite:

- Node 22 install, clone, `npm install` — **`@kmamal/gpu` loads: true**. Its Dawn prebuild does install on a Colab image, which nobody had confirmed.
- lavapipe: `mesa-vulkan-drivers` 23.2.1 installs, and **`shader-f16` is present**.
- `npm run test:kernels` → **32/32 correct, 20 skipped**, `compile_all` 75 shaders clean.
  The skips are all `needs subgroup size >= 32, adapter reports 8` — lavapipe is 8-wide, and the suite says so rather than passing quietly.
  The sampler agrees with the M2 Max token for token (`sampler_wide` → `[116136,116136,116136,116136,95040,95040]` on both), which is cross-backend agreement, not just self-consistency.

**One real bug this run found**, in the notebook *and* in `tests/kernels/gpu.mjs`, which is where the notebook copied it from: Ubuntu 22.04 installs ARCH-SUFFIXED ICD manifests — `lvp_icd.x86_64.json`. The unsuffixed `lvp_icd.json` does not exist, so the loader answers `ERROR_INCOMPATIBLE_DRIVER`, which reads like a missing software rasterizer rather than a wrong filename. With `set -euo pipefail` that killed the cell and cancelled the kernel-suite cell behind it. Both now glob.

**Still not executed here:** the 329 MB pull, the reference, the zip and the download. Those cells remain transcriptions.


## 0. What machine did we get

329 MB of pull, ~370 MB of bundle, and a ~50 MB zip — any Colab tier has room. RAM
matters more than disk: the reference dequantizes six experts at a time (~200 MB), not
all sixty-four (~2.2 GB), so a standard 12 GB instance is comfortable.


In [ ]:
!nproc && free -h | head -2 && df -h /content | tail -1 && (lsb_release -ds || head -2 /etc/os-release)

## 1. Node 22 and the repo

`scripts/pull-tensors.mjs` imports `src/zero-tvm/weight-loader-mlx.ts` directly, so Node
has to strip TypeScript types on import. That is on by default from **22.18**;
`setup_22.x` installs the latest 22.x, which is past it. If the import throws
`Unknown file extension ".ts"`, the Node is older than that — rerun with
`node --experimental-strip-types`.

Colab runs as root, so no `sudo`. If `apt-get install nodejs` complains about a held
package, the image already had a distro `nodejs`; `apt-get remove -y nodejs libnode-dev`
first and repeat.


In [ ]:
%%bash
set -euo pipefail
curl -fsSL https://deb.nodesource.com/setup_22.x | bash - >/dev/null
apt-get install -y -qq nodejs
node -v && npm -v

`PUPPETEER_SKIP_DOWNLOAD` matters: `puppeteer` is a devDependency whose install script
pulls a ~170 MB Chrome that only the e2e suite uses, and nothing in this notebook runs a
browser. `@kmamal/gpu` also has an install script — that one is *not* skippable, it is the
Dawn native binding the kernel suite runs on.


In [ ]:
%%bash
set -euo pipefail
cd /content
[ -d zero-tvm ] || git clone --depth 1 https://github.com/abgnydn/zero-tvm.git
cd zero-tvm
PUPPETEER_SKIP_DOWNLOAD=1 npm install
node -e "console.log('@kmamal/gpu loads:', !!require('@kmamal/gpu'))" 

## 2. lavapipe, so headless WebGPU exists at all

`tests/kernels/gpu.mjs` explains why this is Dawn and not Chrome: Chrome's GPU process
blocklists WebGPU on software rasterizers, so headless WebGPU never initializes on a
machine without a real GPU. Dawn-native has no such blocklist — it talks to whatever
Vulkan ICD is visible, including **lavapipe**, Mesa's CPU device. Its own header gives
exactly these two commands:

```
apt-get install -y mesa-vulkan-drivers
export VK_ICD_FILENAMES=$(ls /usr/share/vulkan/icd.d/lvp_icd*.json | head -1)
```

`VK_ICD_FILENAMES` is not optional decoration: without it the loader finds no ICD on a
GPU-less box and `requestAdapter()` returns null, which surfaces as
`no WebGPU adapter (is a Vulkan ICD visible?)`.


In [ ]:
%%bash
set -euo pipefail
apt-get update -qq
apt-get install -y -qq mesa-vulkan-drivers vulkan-tools
ls -1 /usr/share/vulkan/icd.d/
VK_ICD_FILENAMES=$(ls /usr/share/vulkan/icd.d/lvp_icd*.json | head -1) vulkaninfo --summary 2>/dev/null \
  | sed -n '1,25p'

In [ ]:
%%bash
set -euo pipefail
cd /content/zero-tvm
export VK_ICD_FILENAMES=$(ls /usr/share/vulkan/icd.d/lvp_icd*.json | head -1)
npm run test:kernels

### What that run does and does not establish

**lavapipe is a SOFTWARE rasterizer.** It runs the shaders on the CPU. A green suite here
means the kernels compute the right numbers; it says **nothing whatsoever about
throughput**, and no timing printed by this harness is a measurement of anything.

`gpu.mjs` makes the stronger point that this is not merely a lavapipe caveat: measured on
an M2 Max, `queue.onSubmittedWorkDone()` in this harness resolves on a fixed ~100 ms tick,
so *any* submit costs ~100 ms regardless of its contents — a MoE block timed 100.7 ms
there and 13.4 ms in Chrome for identical work. Time kernels in a browser; use this
harness to decide whether they are right.

Expect some tests to **SKIP loudly**: the `_sg` variants need the WebGPU `subgroups`
feature and a 32-lane subgroup, which lavapipe often does not offer. That is reported, not
silently passed. If `shader-f16` is missing the failure is louder and earlier — most of
the suite needs it.


## 3. Pull one MoE layer, 329 MB

`first_k_dense_replace = 1`, so **layer 0 is a dense mlp at intermediate_size 10944 and
layers 1..26 are MoE at moe_intermediate_size 1408**. Layer 1 is therefore the first MoE
layer and the one to pull. `--match 'layers\.1\.'` cannot catch `layers.11.` — the
trailing `\.` is doing that job.

The dry run first: on a metered or slow link, knowing the number before committing to it
is the whole point of the flag.


In [ ]:
%%bash
set -euo pipefail
cd /content/zero-tvm
node scripts/pull-tensors.mjs mlx-community/DeepSeek-V2-Lite-Chat-4bit-mlx \
  --match 'layers\.1\.' --dry-run

34 tensors, 329.2 MB. The `mlp.switch_mlp.*` stacks are 311 MB of that: three
projections x `[64, N, ...]` weight/scales/biases. Everything else — the whole MLA block,
the router, the shared expert — is 18 MB.

The pull is chunked (8 MB) and **resumable**: a tensor already on disk at the right size
is skipped, so re-running this cell after a disconnect continues instead of restarting.


In [ ]:
%%bash
set -euo pipefail
cd /content/zero-tvm
time node scripts/pull-tensors.mjs mlx-community/DeepSeek-V2-Lite-Chat-4bit-mlx \
  --match 'layers\.1\.' --out .weights-local/kernel-refs/dsv2moe
du -sh .weights-local/kernel-refs/dsv2moe

## 4. The reference — numpy only, deliberately

**mlx does not run here.** There is no Apple silicon on Colab, so a reference that needs
`mx.dequantize` cannot be produced on the machine that has the bandwidth. That is why
`make-dsv2-layer-ref.py` is pure numpy: its `dequant_from()` is the same arithmetic as
`make-mla-ref.py`'s `dequant_numpy`, which `--backend both` proved **bit-identical** to
`mx.dequantize` on real weights — including the narrowing to f16, because a portable path
that is quietly *more* precise than the one it replaces is not portable, it is a second
reference.

The script reads which FFN the layer has off the bundle (`mlp.gate` exists only in a MoE
layer) and dumps every stage, so a failing GPU test localises instead of just being wrong
at the end:

| dump | shape | stage |
|---|---|---|
| `ref_h2` | `[d]` | post-attention norm — the MoE block's input |
| `ref_router_logits` | `[64]` | `h2 @ mlp.gate.weight.T`, **unquantized f16** |
| `ref_router_probs` | `[64]` | softmax over all experts, before top-k |
| `ref_topk_ids` | `[6]` **u32** | descending score, ties to lower index |
| `ref_topk_weights` | `[6]` | the routed weights, **not renormalised** |
| `ref_expert_h` | `[6, 1408]` | `silu(gate)*up` per slot |
| `ref_expert_y` | `[6, d]` | `down(...)` per slot, **unweighted** |
| `ref_expert_out` | `[d]` | the routed sum |
| `ref_shared_h`, `ref_shared_out` | `[2816]`, `[d]` | the shared branch |
| `ref_ffn_out` | `[d]` | routed + shared |
| `ref_out` | `[d]` | residual + FFN |

It asserts as it goes: the softmax sums to 1, the selected experts really are the top-k
(`min selected > max rejected` — the one routing bug with no numerical signature), and
each shipped expert slice really is the expert the router picked, checked against an
independent index into the full stack.


In [ ]:
%%bash
set -euo pipefail
cd /content/zero-tvm
python3 scripts/make-dsv2-layer-ref.py \
  --bundle .weights-local/kernel-refs/dsv2moe --layer model.layers.1

### Read the printed routing line

`top-6 probability mass 0.xx — weights sum to 0.xx (norm_topk_prob=False)` is the number
to look at. DeepSeek-V2 sets `norm_topk_prob: false`, so the six weights are **raw softmax
probabilities and sum to well under 1**. Renormalising them — the default in most
MoE code, and what `moe_router_topk.wgsl` does when `normTopk=1` — rescales the whole
routed branch by `1/mass`. The script prints that factor so the mistake has a size.


## 5. Zip only what has to travel

The rule is one line: **everything except the routed-expert stacks**. Those are the 311 MB
this exercise exists to avoid carrying home, and the reference already sliced the six
experts it used into `exp_*.bin` — 29 MB instead of 311 MB, byte-verbatim from the
checkpoint so a kernel unpacks exactly what the model stores.

What ends up in the zip:

* **reference outputs** (`ref_*.bin`, plus `wk_t`/`wv`, the dequantized halves of
  `kv_b_proj` that the MLA path needs) — ~9 MB
* **6 selected experts of 64** (`exp_{gate,up,down}_proj_{w_u32,s_f16,b_f16}.bin`) — ~29 MB
* **the shared expert** (`mlp.shared_experts.*`, width 2816) — ~10 MB
* **attention + norms** (`q_proj`, `kv_a_proj_with_mqa`, `kv_b_proj`, `o_proj`, three
  layernorms) — ~8 MB
* **the router** (`mlp.gate.weight`, `[64, 2048]` f16, unquantized) — 0.26 MB
* `meta.json`

Measured on a synthetic bundle with these exact shapes: **49.9 MB** zipped from 367 MB on
disk. Quantized nibbles barely compress, so expect the same order on the real thing.

Dropping `exp_*.bin` takes it to ~21 MB and still checks the router, the shared expert and
the combine against `ref_expert_y` — set `INCLUDE_EXPERT_SLICES = False` if the link is
really bad. The routed matmul is then unvalidated.


In [ ]:
import json, pathlib, zipfile

INCLUDE_EXPERT_SLICES = True

B = pathlib.Path("/content/zero-tvm/.weights-local/kernel-refs/dsv2moe")
meta = json.loads((B / "meta.json").read_text())
STACK = f'{meta["layer"]}.mlp.switch_mlp.'

files = [p for p in sorted(B.iterdir())
         if p.is_file() and not p.name.startswith(STACK)
         and (INCLUDE_EXPERT_SLICES or not p.name.startswith("exp_"))]

zpath = pathlib.Path("/content/dsv2moe-refs.zip")
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p in files:
        z.write(p, arcname=f"dsv2moe/{p.name}")


def group(name):
    if name.startswith("ref_") or name in ("wk_t.bin", "wv.bin"):
        return "reference outputs"
    if name.startswith("exp_"):
        return f'{meta["top_k"]} selected experts (of {meta["num_experts"]})'
    if ".self_attn." in name or "layernorm" in name:
        return "attention + norms"
    if ".mlp.gate." in name:
        return "router (unquantized f16)"
    if ".mlp.shared_experts." in name:
        return "shared expert"
    return "meta"


sizes = {}
for p in files:
    g = sizes.setdefault(group(p.name), [0, 0])
    g[0] += 1
    g[1] += p.stat().st_size
for g, (n, b) in sorted(sizes.items(), key=lambda kv: -kv[1][1]):
    print(f"  {g:<30} {n:3d} files {b/1e6:8.2f} MB")

on_disk = sum(p.stat().st_size for p in B.iterdir() if p.is_file())
print(f"\n  bundle on disk {on_disk/1e6:.1f} MB")
print(f"  ZIP            {zpath.stat().st_size/1e6:.1f} MB   {zpath}")
print(f"  routed to experts {meta['selected_experts']} (seed {meta['seed']})")

## 6. Bring it home

`files.download` is fine for 50 MB but dies on a flaky link with no resume. If it does,
mount Drive and copy the zip there instead — Drive's client retries.


In [ ]:
from google.colab import files

files.download("/content/dsv2moe-refs.zip")

# Flaky link? Use Drive instead:
# from google.colab import drive; drive.mount("/content/drive")
# !cp /content/dsv2moe-refs.zip /content/drive/MyDrive/

## 7. On the Mac

```bash
cd ~/dev/zero-tvm
unzip -o ~/Downloads/dsv2moe-refs.zip -d .weights-local/kernel-refs/
npm run test:kernels:real
```

Today that prints:

```
SKIP  dsv2moe   unknown kernel 'dsv2_moe_layer'
```

which is correct and intended. `real-weights.mjs` dispatches on `meta.kernel`, and its
existing `dsv2_layer` handler knows only about MLA — pointing it at a MoE layer would run
the wrong check rather than no check. The bundle is the thing a `dsv2_moe_layer` handler
gets written *against*.

### What that handler is up against — DeepSeek's MoE is not Qwen's

The engine already runs a 7-dispatch MoE block for Qwen3.6. Four things differ here, and
**not one of them raises an error** — each produces a fluent, wrong model:

1. **The router is unquantized.** `mlp.gate.weight` is a plain f16 `[64, 2048]`; there is
   no `mlp.gate.scales` in the checkpoint at all. `moe_router_logits` reads `D/4` u32 words
   per row and its `_q4` twin reads `D/8`; pointed at f16 either one folds neighbouring
   experts' bytes into every logit. **This block needs an f16 matvec that does not exist
   yet.**
2. **The shared expert is not expert-shaped.** `n_shared_experts = 2`, so its width is
   `2 x 1408 = 2816` against a routed expert's 1408. Qwen's shared expert is exactly one
   expert wide, which is what lets the loader stack it as index `E` and lets
   `moe_router_topk` hand it out as slot `K`. This one **cannot be stacked** — it is a
   second, independent matmul chain summed after `moe_combine`.
3. **It has no gate.** No `shared_expert_gate` row, no sigmoid: the shared branch enters at
   weight exactly 1. `moe_router_topk`'s `hasShared` path writes
   `sigmoid(logits[E])` into slot K — wrong here twice over.
4. **`norm_topk_prob` is false.** The six weights are raw softmax probabilities summing to
   well under 1. `moe_router_topk` already supports this (`normTopk = 0` divides by the
   full softmax denominator); the trap is that renormalising is the default everywhere
   else.

Two things that carry over unchanged: `scoring_func` is `softmax` and `topk_method` is
`greedy` with `n_group = topk_group = 1`, so the selection is a plain top-k over all 64 —
the group-limited path never runs. And `routed_scaling_factor` is 1.0, so it is invisible
here; it is applied faithfully anyway, because a DeepSeek that sets it to something else
would otherwise be silently wrong.

The MLA half of this layer is the same shape as the layer-0 bundle's, so the existing
`dsv2_layer` checks port straight over — different weights, same seven stages.
